# Tabelas de Eficiência das Estratégias

Este notebook tem como foco a análise **quantitativa** das simulações.
Ele gera tabelas resumidas agregando os dados de múltiplos experimentos (CSVs), fornecendo estatísticas e números exatos tanto no nível global (treinamento federado como um todo) quanto no nível individual de cada dispositivo.

**Nenhum gráfico é gerado aqui**, apenas DataFrames tabulares prontos para análise.

In [ ]:
import os
import glob
import pandas as pd
from IPython.display import display, HTML

# Configurar o Pandas para exibir todas as linhas e colunas (evitar que sejam truncadas com '...')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 1. Processamento e Extração de Métricas

O código abaixo varre os diretórios e compila duas estruturas de dados agregando a média entre os vários arquivos `.csv`:
1. **Métricas Globais (Rede):** Número de rodadas, tempo total (determinado pelo dispositivo gargalo em cada iteração), acurácia final e soma do consumo de todos os nós.
2. **Métricas Individuais (Nós):** Frequência de seleção absoluta e em porcentagem (taxa de seleção), tempo médio e consumo máximo ao fim do experimento de cada dispositivo.

In [ ]:
base_path = '.'
global_data = []
device_data = []

for dist_folder in ['iid', 'non_iid']:
    dist_path = os.path.join(base_path, dist_folder)
    if not os.path.exists(dist_path):
        continue
        
    subfolders = [f for f in os.listdir(dist_path) if os.path.isdir(os.path.join(dist_path, f))]
    
    for folder in subfolders:
        norm_folder = folder.replace('-', '_')
        distribution = 'IID' if norm_folder.startswith('iid_') else 'Non-IID'
        
        if norm_folder.endswith('_all'): methodology = 'Padrão'
        elif norm_folder.endswith('_leastEnergy'): methodology = 'LeastEnergy'
        elif norm_folder.endswith('_afea'): methodology = 'AFEA'
        elif norm_folder.endswith('_afea_2'): methodology = 'AFEA 2'
        else: continue
            
        csv_files = glob.glob(os.path.join(dist_path, folder, '*metrics*.csv'))
        
        for i, csv_file in enumerate(csv_files):
            try:
                rodada = int(csv_file.split('_')[-1].replace('.csv', ''))
            except ValueError:
                rodada = i + 1
                
            df_temp = pd.read_csv(csv_file)
            training_cols = [c for c in df_temp.columns if c.startswith('training_time_sta')]
            
            # ==========================================
            # MÉTRICAS GLOBAIS (O Treinamento como um todo)
            # ==========================================
            total_iterations = len(df_temp)
            final_acc = df_temp['mean_acc'].iloc[-1] if 'mean_acc' in df_temp.columns else None
            
            # Tempo Global: Em FL síncrono, a iteração só acaba quando o dispositivo selecionado mais lento termina.
            # Portanto o tempo de uma iteração é o .max() entre os selecionados. O tempo global é a soma disso.
            iter_max_times = df_temp[training_cols].max(axis=1)
            total_global_time = iter_max_times.sum()
            
            # Consumo Global: Soma do consumo acumulado final (máximo da rodada) de todos os dispositivos na rede.
            total_energy = 0
            for t_col in training_cols:
                sta_id = t_col.replace('training_time_', '')
                cons_col = f'consumption_{sta_id}'
                if cons_col in df_temp.columns:
                    cons_max = df_temp[cons_col].max()
                    if pd.notnull(cons_max):
                        total_energy += cons_max
                        
            global_data.append({
                'Distribuição': distribution,
                'Metodologia': methodology,
                'Experimento': rodada,
                'Iterações Globais': total_iterations,
                'Acurácia Final': final_acc,
                'Tempo Total da Rede (s)': total_global_time,
                'Consumo Total da Rede (J)': total_energy
            })
            
            # ==========================================
            # MÉTRICAS INDIVIDUAIS (Cada Dispositivo)
            # ==========================================
            for t_col in training_cols:
                sta_id = t_col.replace('training_time_', '')
                cons_col = f'consumption_{sta_id}'
                
                if cons_col in df_temp.columns:
                    valid_mask = pd.notnull(df_temp[t_col])
                    frequencia = valid_mask.sum()
                    taxa_selecao = (frequencia / total_iterations * 100) if total_iterations > 0 else 0
                    
                    if frequencia > 0:
                        tempo_medio = df_temp.loc[valid_mask, t_col].mean()
                        consumo_total = df_temp.loc[valid_mask, cons_col].max()
                    else:
                        tempo_medio = None
                        consumo_total = df_temp[cons_col].max() if pd.notnull(df_temp[cons_col].max()) else 0
                        
                    device_data.append({
                        'Distribuição': distribution,
                        'Metodologia': methodology,
                        'Experimento': rodada,
                        'Dispositivo': sta_id,
                        'Dispositivo_Num': int(sta_id.replace('sta', '')) if 'sta' in sta_id else 999,
                        'Frequência de Seleção': frequencia,
                        'Taxa de Seleção (%)': taxa_selecao,
                        'Tempo Médio (s)': tempo_medio,
                        'Consumo Acumulado (J)': consumo_total
                    })

df_global = pd.DataFrame(global_data)
df_device = pd.DataFrame(device_data)

print("Processamento concluído. Matrizes de dados extraídas com sucesso.")

## 2. Eficiência do Treinamento Federado (Visão Global)

Esta tabela responde a perguntas sobre o comportamento sistêmico da estratégia. Os valores são a **média** calculada entre todos os experimentos/CSVs da mesma metodologia.

- **Iterações Globais:** Número total de rodadas de comunicação para convergir (linhas do CSV).
- **Acurácia Final:** Acurácia global obtida no modelo ao fim do treino.
- **Tempo Total da Rede (s):** Considerando o modelo síncrono, o tempo da rede é pautado pelo nó escolhido mais lento em cada rodada (Gargalo). Este é o tempo somado de uma simulação real.
- **Consumo Total da Rede (J):** Soma da energia gasta por todos os dispositivos do sistema.

In [ ]:
tabela_global = df_global.groupby(['Distribuição', 'Metodologia']).agg({
    'Experimento': 'count', # Conta a amostra de arquivos lidos
    'Iterações Globais': 'mean',
    'Acurácia Final': 'mean',
    'Tempo Total da Rede (s)': 'mean',
    'Consumo Total da Rede (J)': 'mean'
}).reset_index().rename(columns={'Experimento': 'Qtd de Simulações'}).round(2)

display(HTML(tabela_global.to_html(index=False)))

## 3. Eficiência por Dispositivo (Visão Individual)
Abaixo geramos tabelas comparativas para cada métrica individual (colocando as metodologias lado a lado nas colunas), facilitando o estudo do impacto sobre um nó específico (ex: `sta0`). Os valores são a **média entre os experimentos**.

In [ ]:
# Primeiro, geramos a tabela base (médias)
tabela_device_mean = df_device.groupby(['Distribuição', 'Metodologia', 'Dispositivo', 'Dispositivo_Num']).agg({
    'Taxa de Seleção (%)': 'mean',
    'Frequência de Seleção': 'mean',
    'Tempo Médio (s)': 'mean',
    'Consumo Acumulado (J)': 'mean'
}).reset_index().sort_values(['Distribuição', 'Dispositivo_Num'])

# Função para pivotar a tabela de forma que as Metodologias fiquem nas colunas, 
# gerando uma tabela altamente comparativa
def exibir_tabela_comparativa(df, metrica, titulo, casas_decimais=2):
    pivot = df.pivot_table(index=['Distribuição', 'Dispositivo'], columns='Metodologia', values=metrica).round(casas_decimais)
    
    # Ordenar as colunas logicamente para facilitar a leitura se elas existirem
    cols_order = [c for c in ['Padrão', 'LeastEnergy', 'AFEA', 'AFEA 2'] if c in pivot.columns]
    pivot = pivot[cols_order]
    
    html = f"<h3>{titulo}</h3>" + pivot.to_html()
    display(HTML(html))
    print("\n")

# Tabela 1: Taxa de Seleção (Participação percentual nas rodadas globais)
exibir_tabela_comparativa(tabela_device_mean, 'Taxa de Seleção (%)', 'Tabela Comparativa: Taxa Percentual de Seleção (%)')

# Tabela 2: Frequência Absoluta de Seleção
exibir_tabela_comparativa(tabela_device_mean, 'Frequência de Seleção', 'Tabela Comparativa: Qtd. Absoluta de Seleções por Experimento')

# Tabela 3: Tempo de Treinamento
exibir_tabela_comparativa(tabela_device_mean, 'Tempo Médio (s)', 'Tabela Comparativa: Tempo Médio Gasto Treinando (s)', casas_decimais=4)

# Tabela 4: Consumo Total/Acumulado da Simulação
exibir_tabela_comparativa(tabela_device_mean, 'Consumo Acumulado (J)', 'Tabela Comparativa: Consumo Energético Total Requerido (J)')